In [ ]:
#Libraries
import pandas as pd
import re
import html
import unicodedata

In [ ]:
#Load the dataset
df = pd.read_csv("/content/Phishing_Email.csv")

print("Original dataset:", df.shape)

Original dataset: (18650, 3)


In [ ]:
#Keep the required columns only
df = df[["Email Text", "Email Type"]].copy()

In [ ]:
#Remove empty and misssing email texts
df["Email Text"] = df["Email Text"].fillna("").astype(str)

df["Email Text"] = df["Email Text"].str.strip()

df = df[df["Email Text"] != ""].copy()

print("After removing empty/missing emails:", df.shape)

After removing empty/missing emails: (18631, 2)


In [ ]:
#Remove emails with different label
label_counts = (
    df.groupby("Email Text")["Email Type"]
      .nunique()
)

conflicting_texts = label_counts[label_counts > 1].index

print("Conflicting email texts:", len(conflicting_texts))

df = df[~df["Email Text"].isin(conflicting_texts)].copy()

Conflicting email texts: 1


In [ ]:
#Remove duplicate emails
before_duplicates = len(df)

df = df.drop_duplicates(
    subset=["Email Text"],
    keep="first"
).copy()

print("Duplicate emails removed:", before_duplicates - len(df))

Duplicate emails removed: 577


In [ ]:
#NLP text normalisation
def preprocess_email(text):

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Unicode normalisation
    text = unicodedata.normalize("NFKC", text)

    # Remove NULL/control characters
    text = "".join(
        char for char in text
        if char == "\n" or char == "\t" or not unicodedata.category(char).startswith("C")
    )

    # Remove HTML/XML tags while preserving their visible text
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalise common escape sequences
    text = text.replace("\\n", "\n")
    text = text.replace("\\r", "\n")
    text = text.replace("\\t", " ")

    # Normalise line breaks
    text = re.sub(r"\r\n|\r", "\n", text)

    # Collapse excessive whitespace
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Remove leading/trailing whitespace
    text = text.strip()
    return text


df["processed_text"] = df["Email Text"].apply(preprocess_email)

In [ ]:
#Check and remove email with empty texts
df["processed_text"] = df["processed_text"].fillna("").str.strip()

df = df[df["processed_text"] != ""].copy()

In [ ]:
#Label encoding
df["label"] = df["Email Type"].map({
    "Safe Email": 0,
    "Phishing Email": 1
})


# Check that every label was converted
print("\nUnmapped labels:", df["label"].isna().sum())

df = df.dropna(subset=["label"]).copy()

df["label"] = df["label"].astype(int)


Unmapped labels: 0


In [ ]:
#Final dataset
final_df = df[
    ["Email Text", "processed_text", "Email Type", "label"]
].reset_index(drop=True)

In [ ]:
#Final dataset inspection
print("\n" + "=" * 60)
print("FINAL PROCESSED DATASET")
print("=" * 60)

print("Shape:", final_df.shape)

print("\nClass distribution:")
print(final_df["Email Type"].value_counts())

print("\nLabel distribution:")
print(final_df["label"].value_counts())

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nDuplicate processed texts:")
print(final_df["processed_text"].duplicated().sum())


FINAL PROCESSED DATASET
Shape: (17521, 4)

Class distribution:
Email Type
Safe Email        10978
Phishing Email     6543
Name: count, dtype: int64

Label distribution:
label
0    10978
1     6543
Name: count, dtype: int64

Missing values:
Email Text        0
processed_text    0
Email Type        0
label             0
dtype: int64

Duplicate processed texts:
2


In [ ]:
#Remove duplicate processed texts
before = len(final_df)

final_df = final_df.drop_duplicates(
    subset=["processed_text"],
    keep="first"
).reset_index(drop=True)

print("Processed duplicates removed:", before - len(final_df))
print("Final dataset shape:", final_df.shape)

print("\nFinal class distribution:")
print(final_df["Email Type"].value_counts())

print("\nFinal label distribution:")
print(final_df["label"].value_counts())

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nRemaining duplicate processed texts:",
      final_df["processed_text"].duplicated().sum())

Processed duplicates removed: 2
Final dataset shape: (17519, 4)

Final class distribution:
Email Type
Safe Email        10978
Phishing Email     6541
Name: count, dtype: int64

Final label distribution:
label
0    10978
1     6541
Name: count, dtype: int64

Missing values:
Email Text        0
processed_text    0
Email Type        0
label             0
dtype: int64

Remaining duplicate processed texts: 0


In [ ]:
#Save the final dataset
output_path = "/content/phishing_email_processed.csv"

final_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("\nFinal dataset is ready")
print("Saved to:", output_path)


Final dataset is ready
Saved to: /content/phishing_email_processed.csv
